# 28 — IG vs SHAP: Most Divergent and Most Aligned Cases — Helpdesk

Scans prefixes and ranks them by how much IG and SHAP attribution profiles
diverge. Shows the top-X cases with the **highest** divergence and the
top-X with the **lowest** divergence, with side-by-side heatmaps.

In [ ]:
import sys
from pathlib import Path
_current = Path().resolve()
while _current != _current.parent:
    if (_current / 'src').is_dir(): break
    _current = _current.parent
if str(_current) not in sys.path:
    sys.path.insert(0, str(_current))
if str(_current / 'src') not in sys.path:
    sys.path.insert(0, str(_current / 'src'))

In [ ]:
import random
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from tqdm.notebook import tqdm
from scipy.stats import spearmanr

from src.interpretability.config.helpdesk_config import CONFIG
from src.model.dropout_uncertainty_enc_dec_LSTM.dropout_uncertainty_model import DropoutUncertaintyEncoderDecoderLSTM
from src.evaluation.evaluation import Evaluation
from src.interpretability import InterpretabilityTool
from src.interpretability.attribution.shap_explainer import SequenceSHAP

# --- Config ---
TEST_MODE = True
SCAN_SIZE = 60        # how many prefixes to scan
TOP_X = 3             # how many extreme cases to show
IG_STEPS = 30
SHAP_SAMPLES = 30
SEED = 42
random.seed(SEED)

TARGET = CONFIG.concept_name
%matplotlib inline

In [ ]:
model = DropoutUncertaintyEncoderDecoderLSTM.load(str(CONFIG.get_model_path()), dropout=0.0)
model.eval()
test_dataset = torch.load(str(CONFIG.get_test_data_path()), weights_only=False)
eval_helper = Evaluation(
    model=model, dataset=test_dataset, concept_name=CONFIG.concept_name,
    growing_num_values=CONFIG.growing_num_values, all_cat=CONFIG.all_cat, all_num=CONFIG.all_num)
tool = InterpretabilityTool(model, model.data_set_categories)
shap_explainer = SequenceSHAP(model, model.data_set_categories)
print(f'Cases: {len(eval_helper.cases)}')

In [ ]:
all_samples = []
for case_name, full_case in eval_helper.cases.items():
    for prefix_len, prefix, suffix in eval_helper._iterate_case(full_case):
        if prefix_len >= 2:
            all_samples.append({'case_name': case_name, 'prefix_len': prefix_len, 'prefix': prefix})

if TEST_MODE:
    samples = random.sample(all_samples, min(SCAN_SIZE, len(all_samples)))
else:
    samples = all_samples
print(f'Scanning {len(samples)} prefixes')

## Scan: compute IG and SHAP for each prefix and measure divergence

In [ ]:
results = []

for sample in tqdm(samples, desc='IG + SHAP scan'):
    prefix = sample['prefix']
    prefix_len = sample['prefix_len']
    cat_t = [t.squeeze(0) if t.dim() > 1 else t for t in prefix[0]]
    num_t = [t.squeeze(0) if t.dim() > 1 else t for t in prefix[1]]
    process = (cat_t, num_t)

    try:
        attr_ig = tool.compute_attribution_map(
            process=process, prefix_length=prefix_len,
            target=TARGET, target_class='auto',
            method='integrated_gradients', n_steps=IG_STEPS)

        attr_shap = tool.compute_attribution_map(
            process=process, prefix_length=prefix_len,
            target=TARGET, target_class='auto',
            method='shap', n_samples=SHAP_SAMPLES)

        # Feature-level |attribution| totals
        ig_vec = []
        shap_vec = []
        for feat in attr_ig.feature_names:
            ig_v = attr_ig.attributions.get(feat, np.zeros(1))
            sh_v = attr_shap.attributions.get(feat, np.zeros(1))
            if hasattr(ig_v, 'detach'): ig_v = ig_v.detach().cpu().numpy()
            if hasattr(sh_v, 'detach'): sh_v = sh_v.detach().cpu().numpy()
            ig_vec.append(float(np.abs(ig_v).sum()))
            shap_vec.append(float(np.abs(sh_v).sum()))

        ig_arr = np.array(ig_vec)
        shap_arr = np.array(shap_vec)

        # Normalize to sum=1 for fair comparison
        ig_norm = ig_arr / ig_arr.sum() if ig_arr.sum() > 0 else ig_arr
        shap_norm = shap_arr / shap_arr.sum() if shap_arr.sum() > 0 else shap_arr

        # Divergence = L1 distance between normalized profiles
        divergence = float(np.abs(ig_norm - shap_norm).sum())
        rho, _ = spearmanr(ig_arr, shap_arr)

        results.append({
            'case_name': sample['case_name'],
            'prefix_len': prefix_len,
            'divergence': divergence,
            'spearman': rho,
            'attr_ig': attr_ig,
            'attr_shap': attr_shap,
            'process': process,
        })
    except Exception as e:
        print(f'Error {sample["case_name"]}: {e}')

results.sort(key=lambda r: r['divergence'])
print(f'\nScanned {len(results)} prefixes')
print(f'Divergence range: {results[0]["divergence"]:.4f} (most aligned) to {results[-1]["divergence"]:.4f} (most divergent)')

## Divergence distribution

In [ ]:
divs = [r['divergence'] for r in results]
rhos = [r['spearman'] for r in results]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ax = axes[0]
ax.hist(divs, bins=20, color='steelblue', edgecolor='white')
ax.set_xlabel('L1 divergence (normalized profiles)')
ax.set_ylabel('Count')
ax.set_title(f'IG-SHAP Divergence Distribution\n(mean={np.mean(divs):.3f})')

ax = axes[1]
ax.hist(rhos, bins=20, color='darkorange', edgecolor='white')
ax.set_xlabel('Spearman correlation')
ax.set_ylabel('Count')
ax.set_title(f'IG-SHAP Spearman Distribution\n(mean={np.nanmean(rhos):.3f})')

plt.suptitle('How much do IG and SHAP disagree across cases?', fontweight='bold')
plt.tight_layout()
plt.show()

## Most ALIGNED cases (lowest divergence)

In [ ]:
def plot_case_comparison(r, rank_label):
    """Plot IG, SHAP, and delta heatmaps for one case."""
    attr_ig = r['attr_ig']
    attr_shap = r['attr_shap']
    features = attr_ig.feature_names
    n_f = len(features)
    n_s = len(attr_ig.step_labels)

    def to_mat(am):
        mat = np.zeros((n_f, n_s))
        for i, feat in enumerate(features):
            if feat in am.attributions:
                v = am.attributions[feat]
                if hasattr(v, 'detach'): v = v.detach().cpu().numpy()
                v = np.asarray(v).flatten()
                mat[i, :min(len(v), n_s)] = v[:n_s]
        return mat

    m_ig = to_mat(attr_ig)
    m_sh = to_mat(attr_shap)
    m_s = min(m_ig.shape[1], m_sh.shape[1])
    m_ig, m_sh = m_ig[:, :m_s], m_sh[:, :m_s]
    m_delta = m_ig - m_sh
    steps = attr_ig.step_labels[:m_s]

    vmax = max(np.abs(m_ig).max(), np.abs(m_sh).max(), 1e-6)
    vmax_d = max(np.abs(m_delta).max(), 1e-6)

    fig, axes = plt.subplots(1, 3, figsize=(18, max(4, n_f * 0.3)))
    for ax, mat, title, vm in zip(
            axes, [m_ig, m_sh, m_delta],
            ['IG', 'SHAP', 'Delta (IG-SHAP)'],
            [vmax, vmax, vmax_d]):
        norm = TwoSlopeNorm(vmin=-vm, vcenter=0, vmax=vm)
        im = ax.imshow(mat, cmap='RdBu_r', aspect='auto', norm=norm, interpolation='nearest')
        ax.set_xticks(range(len(steps)))
        ax.set_xticklabels(steps, rotation=45, ha='right', fontsize=7)
        ax.set_yticks(range(n_f))
        ax.set_yticklabels(features, fontsize=7)
        ax.set_title(title, fontsize=10, fontweight='bold')
        fig.colorbar(im, ax=ax, shrink=0.6)

        if mat.shape[0] * mat.shape[1] <= 80:
            for i in range(mat.shape[0]):
                for j in range(mat.shape[1]):
                    v = mat[i, j]
                    c = 'white' if abs(v) > vm * 0.5 else 'black'
                    ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=6, color=c)

    axes[0].set_ylabel('Features')
    fig.suptitle(
        f'{rank_label} | {r["case_name"]} | prefix={r["prefix_len"]}\n'
        f'divergence={r["divergence"]:.4f}, spearman={r["spearman"]:.3f} | '
        f'{attr_ig.target_description}',
        fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()


aligned = results[:TOP_X]
for i, r in enumerate(aligned):
    plot_case_comparison(r, f'ALIGNED #{i+1}')

## Most DIVERGENT cases (highest divergence)

In [ ]:
divergent = results[-TOP_X:][::-1]
for i, r in enumerate(divergent):
    plot_case_comparison(r, f'DIVERGENT #{i+1}')

## Summary table

In [ ]:
rows = []
for label, group in [('ALIGNED', aligned), ('DIVERGENT', divergent)]:
    for i, r in enumerate(group):
        # Top IG and SHAP features
        ig_feats = {f: float(np.abs(np.asarray(r['attr_ig'].attributions.get(f, [0]))).sum())
                    for f in r['attr_ig'].feature_names}
        shap_feats = {f: float(np.abs(np.asarray(r['attr_shap'].attributions.get(f, [0]))).sum())
                      for f in r['attr_shap'].feature_names}
        top_ig = max(ig_feats, key=ig_feats.get)
        top_shap = max(shap_feats, key=shap_feats.get)
        rows.append({
            'Group': label,
            'Rank': i + 1,
            'Case': r['case_name'],
            'Prefix': r['prefix_len'],
            'Divergence': f'{r["divergence"]:.4f}',
            'Spearman': f'{r["spearman"]:.3f}',
            'Top IG Feature': top_ig,
            'Top SHAP Feature': top_shap,
            'Same Top?': top_ig == top_shap,
        })

display(pd.DataFrame(rows))